# Download Sentinel-1 Partial Products By Product ID

Download a chosen list of Sentinel-1 SLC products as cropped `.partial.SAFE` products. The STAC query uses `ids=product_ids`, while the AOI controls which consecutive bursts are written into each cropped measurement TIFF.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import json
import logging

import geopandas as gpd
import yaml

from eo_tools.S1.download import download_partial_products, search_products

logging.basicConfig(level=logging.INFO)
log = logging.getLogger(__name__)

In [ ]:
# Keep partial products alongside those downloaded by download-partial-product-s1.ipynb.
data_dir = "/data/S1/partial_dls/"

# Use the S3 credentials created on the CDSE website.
with open("/data/creds_s3.json") as f:
    cred = json.load(f)

## Select Products By ID

Replace or extend this list with Sentinel-1 SLC item IDs from the CDSE STAC catalog. IDs do not include the `.SAFE` suffix.

In [ ]:
# This AOI selects the burst subset downloaded from each product.
aoi_file = "../data/Morocco_AOI.geojson"
shp = gpd.read_file(aoi_file).geometry[0]

product_ids = [
    "S1A_IW_SLC__1SDV_20230904T063730_20230904T063757_050174_0609E3_DAA1",
    "S1A_IW_SLC__1SDV_20230916T063730_20230916T063757_050349_060FCD_6814",
]

products = search_products(
    intersects=shp,
    ids=product_ids,
)

found_ids = set(products.id)
missing_ids = [product_id for product_id in product_ids if product_id not in found_ids]
if missing_ids:
    raise RuntimeError(f"CDSE STAC did not return requested product IDs: {missing_ids}")

# Preserve the requested order for download messages and output inspection.
products = products.set_index("id").loc[product_ids].reset_index()

outside_aoi = products[~products.intersects(shp)].id.tolist()
if outside_aoi:
    raise RuntimeError(f"Requested products do not intersect the download AOI: {outside_aoi}")

products[["id", "startTimeFromAscendingNode", "relativeOrbitNumber", "orbitDirection"]]

## Download Partial SAFE Products

Set `pol` to `"vv"`, `"vh"`, or `"full"`. Leave `force_overwrite=False` to avoid replacing an existing partial product unless its contents should be rebuilt.

In [ ]:
download_partial_products(
    products,
    shp,
    out_dir=data_dir,
    aws_key=cred["username"],
    aws_secret=cred["password"],
    pol="full",
    force_overwrite=True,
)

## Inspect Requested Partial Products

In [ ]:
partial_products = [Path(data_dir) / f"{product_id}.partial.SAFE" for product_id in product_ids]

for product_dir in partial_products:
    if not product_dir.exists():
        print(f"Missing output: {product_dir}")
        continue

    manifest_file = product_dir / "partial_download.yml"
    with manifest_file.open() as f:
        manifest = yaml.safe_load(f)

    print("-" * 80)
    print(product_dir.name)
    print(f"  manifest: {manifest_file}")
    for subswath, polarizations in manifest["subsets"].items():
        for pol, subset in polarizations.items():
            print(
                f"  {subswath.upper()} / {pol.upper()}: "
                f"bursts {subset['min_burst']} to {subset['max_burst']}, "
                f"lines {subset['number_of_lines']}"
            )